# 6장 실습 ① — 과적합은 언제 생기는가

**TensorFlow 판**

5장 끝에서 학습 손실을 0.0000까지 떨어뜨렸습니다. **그것이 좋은 일일까요.**

본문 §6.1과 §6.2의 표를 이 노트북이 만듭니다.

## 6.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 6.1 실험대 — 잡음을 키운 소용돌이

5장에서는 잡음 0.06을 썼습니다. 여기서는 **0.25**로 키웁니다.
잡음이 있어야 *"잡음까지 외우는"* 현상이 보이기 때문입니다.

In [ ]:
def make(n, seed=42, noise=0.25):
    """잡음을 0.25로 키운다 — 과적합이 눈에 보이도록."""
    x, y = data.spirals(n, seed=seed, noise=noise)
    return data.split(x, y, val_ratio=0.2, test_ratio=0.2, seed=seed)

s = make(1600)
print(s.summary())
plot.scatter2d(s.x_train, s.y_train, class_names=("무리 0", "무리 1"),
               xlabel="$x_1$", ylabel="$x_2$", title="잡음 0.25의 소용돌이")
plt.show()

## 6.2 학습 함수 — 여기만 판마다 다릅니다

드롭아웃·L2·배치 정규화·조기 종료를 옵션으로 받습니다.
**이 아래의 모든 셀은 세 판이 같습니다.**

In [ ]:
import tensorflow as tf

dlbook.set_seed(42)

def train(split, units=(256, 256, 256), drop=0.0, l2=0.0, bn=False,
          act="relu", lr=0.001, epochs=200, bs=32, early=0, seed=42):
    """모델을 만들어 학습시키고 (학습 정확도, 시험 정확도, history)를 돌려준다.

    이 함수 하나만 판마다 다르다. 아래의 모든 실험 셀은 세 판이 같다.
    """
    dlbook.set_seed(seed)
    reg = tf.keras.regularizers.l2(l2) if l2 else None
    ls = [tf.keras.layers.Input(shape=(2,))]
    for u in units:
        ls.append(tf.keras.layers.Dense(u, activation=None if bn else act,
                                        kernel_regularizer=reg))
        if bn:
            ls += [tf.keras.layers.BatchNormalization(),
                   tf.keras.layers.Activation(act)]
        if drop:
            ls.append(tf.keras.layers.Dropout(drop))
    ls.append(tf.keras.layers.Dense(1, activation="sigmoid"))
    model = tf.keras.Sequential(ls)
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    cbs = ([tf.keras.callbacks.EarlyStopping(patience=early,
                                             restore_best_weights=True)]
           if early else [])
    h = model.fit(split.x_train, split.y_train,
                  validation_data=(split.x_val, split.y_val),
                  epochs=dlbook.smoke.epochs(epochs), batch_size=bs,
                  verbose=0, callbacks=cbs)
    def acc(xa, ya):
        p = (model.predict(xa, verbose=0).reshape(-1) > 0.5).astype("int64")
        return metrics.accuracy(ya, p)
    return acc(split.x_train, split.y_train), acc(split.x_test, split.y_test), h.history

## 6.3 데이터 양만 바꿔 봅니다

**과적합의 가장 큰 원인은 데이터 부족입니다.**
모델은 그대로 두고 데이터만 늘려 봅니다.

In [ ]:
# 모델은 그대로 두고 데이터 양만 바꾼다.
sizes = [60, 150, 400] if dlbook.smoke.is_smoke() else [60, 150, 400, 1600]
print(f"{'학습 데이터':<12}{'학습 정확도':>12}{'시험 정확도':>12}{'차이':>10}")
gaps = {}
for n in sizes:
    sn = make(n)
    tr, te, h = train(sn)
    gaps[n] = (tr, te)
    print(f"{len(sn.x_train):<12}{tr:>12.3f}{te:>12.3f}{tr - te:>10.3f}")
    dlbook.record(f"ch06_n{n}_train_acc", tr)
    dlbook.record(f"ch06_n{n}_test_acc", te)

print()
print("→ 모델을 손대지 않았습니다. 데이터만 늘렸습니다.")
print("→ 학습 정확도가 떨어지고 시험 정확도가 오릅니다. 그것이 좋은 신호입니다.")

## 6.4 손실 곡선에서 갈라지는 지점

과적합은 학습 손실만 보면 알 수 없습니다.
**검증 손실이 방향을 트는 지점**이 신호입니다.

In [ ]:
# 데이터가 적을 때 두 곡선이 갈라지는 것을 눈으로 봅니다.
s_small = make(150)
tr, te, h = train(s_small)
plot.loss_curve(h, title=f"학습 데이터 {len(s_small.x_train)}개 — 갈라지는 지점을 찾으십시오")
plt.show()
print(f"학습 {tr:.3f} / 시험 {te:.3f} / 차이 {tr - te:.3f}")

## 정리

- **과적합은 잡음까지 외운 상태**입니다. 학습 손실만 보면 알 수 없습니다.
- **데이터를 늘리면 학습 정확도가 떨어지고 시험 정확도가 오릅니다.**
  모델을 손대지 않아도 그렇습니다.
- 4장에서 말한 *"백 단위는 거의 무의미하다"* 의 근거가 이 표입니다.

### 연습

1. 잡음을 0.05로 낮추면 표가 어떻게 달라집니까. 왜 그렇습니까.
2. 데이터를 그대로 두고 **모델을 (32, 32, 32)로 줄이면** 차이가 어떻게 됩니까.
3. 학습 데이터 60개에서 손실 곡선을 그려, 몇 epoch에서 갈라지는지 찾으십시오.